# 爬證交所資料

In [1]:
import pandas as pd
import numpy as np
import json
import requests
import time
import os

# folder_list = os.listdir(os.path.dirname(os.path.abspath(os.getcwd())) + "/stock_news")
# folder_list = [folder_list for folder_list in folder_list if folder_list != '.DS_Store']
# print(folder_list)

# 建立一個字典，其中包含所有欄位名稱的中文到英文的映射
column_name_mapping = {
    '日期': 'Date',
    '成交股數': 'Volume',
    '成交金額': 'Transaction Amount',
    '開盤價': 'Open',
    '最高價': 'High',
    '最低價': 'Low',
    '收盤價': 'Close',
    '漲跌價差': 'Diff',
    '成交筆數': 'Transaction Count'
}

def fetch_data_twse(ticker_symbol, year1, year2):
    print("Fetching data for " + ticker_symbol + " from " + str(year1) + " to " + str(year2) + "...")
    df_y = pd.DataFrame()
    for i in range(year1, year2+1):
        for j in range(12):
            month = str(i) + str(j+1).zfill(2)
            url = f'https://www.twse.com.tw/exchangeReport/STOCK_DAY?response=json&date={month}01&stockNo={ticker_symbol}'
            print(url)
            data = requests.get(url).text
            data = json.loads(data)
            if data['stat'] != 'OK':
                print('Data not found for ' + month)
                break
            df_m = pd.DataFrame(data['data']) 
            df_m.columns = data['fields']
            df_m = df_m.rename(columns=column_name_mapping)
            df_m.set_index('Date', inplace=True)
            df_y = pd.concat([df_y, df_m])
            print(df_y)
            # time.sleep(1)

    # 將日期從民國年轉換為西元年
    df_y.index = pd.Series(df_y.index).str.replace('/', '').apply(lambda x: str(int(x[:3]) + 1911) + x[3:])

    return df_y


In [3]:
path = os.path.join(os.path.abspath(os.path.dirname(os.getcwd())), 'history_data', 'tw', 'stock_price')
# folder_list = os.listdir(os.path.dirname(os.path.abspath(os.getcwd())) + "/stock_news")
# folder_list = [folder_list for folder_list in folder_list if folder_list != '.DS_Store']
# print(folder_list)
folder_list = ['2454', '2317', '2382', '2308', '3711']

for id in folder_list:
    try:
        stock_tech = fetch_data_twse(id, 2022, 2024) # 這裡可以自行設定要抓取的時間範圍
        stock_tech.to_csv(os.path.join(path , id + 'twse.csv')) # 將資料存成 csv 檔
        print('save ' + id + 'twse.csv')
    except Exception as e:
        print("Error: " + id + " " + str(e))
        pass

Fetching data for 2454 from 2022 to 2024...
https://www.twse.com.tw/exchangeReport/STOCK_DAY?response=json&date=20220101&stockNo=2454
              Volume Transaction Amount      Open      High       Low  \
Date                                                                    
111/01/03  7,574,367      8,840,518,430  1,205.00  1,215.00  1,145.00   
111/01/04  6,761,854      7,830,403,615  1,165.00  1,175.00  1,145.00   
111/01/05  7,170,388      8,393,883,150  1,170.00  1,190.00  1,150.00   
111/01/06  8,411,284      9,514,641,165  1,150.00  1,155.00  1,110.00   
111/01/07  8,690,396      9,587,680,095  1,130.00  1,145.00  1,090.00   
111/01/10  7,452,615      8,071,954,585  1,080.00  1,095.00  1,070.00   
111/01/11  4,807,940      5,188,806,882  1,090.00  1,095.00  1,065.00   
111/01/12  3,084,290      3,337,695,090  1,095.00  1,095.00  1,070.00   
111/01/13  5,091,825      5,575,476,675  1,090.00  1,110.00  1,085.00   
111/01/14  5,753,681      6,152,838,075  1,090.00  1,090.00  1,

# 畫技術分析


In [14]:
import pandas as pd
import numpy as np
import json
import requests
import time
import os

path = os.path.join(os.path.abspath(os.path.dirname(os.getcwd())), 'history_data', 'tw', 'stock_price')
# folder_list = ['2330', '2454', '2317', '2382', '2308', '3711']


id = '3711'
stock_tech = pd.read_csv(os.path.join(path , id + 'twse.csv'))

# # 將stock_tech的值都換成float
# stock_tech['Open'] = stock_tech['Open'].map(lambda x: float(x.replace(',', '')))
# stock_tech['Close'] = stock_tech['Close'].map(lambda x: float(x.replace(',', '')))
# stock_tech['High'] = stock_tech['High'].map(lambda x: float(x.replace(',', '')))
# stock_tech['Low'] = stock_tech['Low'].map(lambda x: float(x.replace(',', '')))


# 計算移動均線 (SMA)
stock_tech['MA5'] = stock_tech['Close'].rolling(window=5).mean()
stock_tech['MA10'] = stock_tech['Close'].rolling(window=10).mean()
stock_tech['MA20'] = stock_tech['Close'].rolling(window=20).mean()

# 計算指數移動平均線 (EMA)
stock_tech['EMA5'] = stock_tech['Close'].ewm(span=5, adjust=False).mean()
stock_tech['EMA12'] = stock_tech['Close'].ewm(span=12, adjust=False).mean()
stock_tech['EMA26'] = stock_tech['Close'].ewm(span=26, adjust=False).mean()

# 計算MACD和DIF
stock_tech['DIF'] = stock_tech['EMA12'] - stock_tech['EMA26']
stock_tech['MACD'] = stock_tech['DIF'].ewm(span=9, adjust=False).mean()


# 計算RSI
def calculate_rsi(series, period=14):
    delta = series.diff(1)
    gain = (delta.where(delta > 0, 0)).rolling(window=period).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=period).mean()
    rs = gain / loss
    rsi = 100 - (100 / (1 + rs))
    return rsi

stock_tech['RSI'] = calculate_rsi(stock_tech['Close'])

# 計算KDJ
low_list = stock_tech['Low'].rolling(9, min_periods=9).min()
low_list.fillna(value=stock_tech['Low'].expanding().min(), inplace=True)
high_list = stock_tech['High'].rolling(9, min_periods=9).max()
high_list.fillna(value=stock_tech['High'].expanding().max(), inplace=True)
rsv = (stock_tech['Close'] - low_list) / (high_list - low_list) * 100


# 初始值設定
stock_tech['K'] = 50
stock_tech['D'] = 50

for i in range(1, len(stock_tech)):
    stock_tech.loc[stock_tech.index[i], 'K'] = (2/3) * stock_tech.loc[stock_tech.index[i-1], 'K'] + (1/3) * rsv[i]
    stock_tech.loc[stock_tech.index[i], 'D'] = (2/3) * stock_tech.loc[stock_tech.index[i-1], 'D'] + (1/3) * stock_tech.loc[stock_tech.index[i], 'K']


# 顯示DataFrame
stock_tech.to_csv(os.path.join(path , id + 'tech.csv'), encoding='utf-8', index=False)

/var/folders/xc/rjj8m66j6xs5pgcp1hc7g2kr0000gn/T/ipykernel_88501/3297899851.py:61: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '59.99999999999999' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  stock_tech.loc[stock_tech.index[i], 'K'] = (2/3) * stock_tech.loc[stock_tech.index[i-1], 'K'] + (1/3) * rsv[i]
/var/folders/xc/rjj8m66j6xs5pgcp1hc7g2kr0000gn/T/ipykernel_88501/3297899851.py:62: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '53.33333333333333' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  stock_tech.loc[stock_tech.index[i], 'D'] = (2/3) * stock_tech.loc[stock_tech.index[i-1], 'D'] + (1/3) * stock_tech.loc[stock_tech.index[i], 'K']
